# 02 · Análise Exploratória de Dados (EDA)

Roda em cima de `data/processed/features_farl.parquet` (gerado no notebook 01). Não precisa de GPU.

Pode rodar no Colab (o bloco abaixo pega o arquivo do Drive) ou localmente no VS Code.
Todas as figuras são salvas em `reports/figures/` para irem direto ao relatório.

In [ ]:
import os, sys
EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/Rodrigotorres1/veridade.git"
    if not os.path.exists("/content/repo"):
        os.system(f"git clone -q {REPO_URL} /content/repo")
    os.chdir("/content/repo")
    os.makedirs("data/processed", exist_ok=True)
    os.system("cp /content/drive/MyDrive/projeto-ml/processed/features_farl.parquet data/processed/")
else:
    os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import (FEATURES_FILE, EMB_COLS, LIMIARES_CONTRIBUICAO, LIMIAR_PRINCIPAL, ROOT)

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
PALETA = {"UTKFace": "#2E6F9E", "FG-NET": "#D9822B"}

def salvar(nome):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{nome}.png", dpi=200, bbox_inches="tight")
    plt.show()

df = pd.read_parquet(FEATURES_FILE)
meta = df.drop(columns=EMB_COLS)
E = df[EMB_COLS].to_numpy()
print(df.shape)

## 1. Visão geral e estatísticas descritivas

In [ ]:
resumo = meta.groupby("dataset").agg(
    imagens=("idade", "size"),
    idade_media=("idade", "mean"), idade_mediana=("idade", "median"),
    idade_desvio=("idade", "std"), idade_min=("idade", "min"), idade_max=("idade", "max"),
    pct_menor_18=("idade", lambda s: 100 * (s < 18).mean()),
).round(2)
resumo

In [ ]:
print("Imagens por split:")
print(pd.crosstab(meta["dataset"], meta["split"], margins=True))
print()
print("Sujeitos distintos no FG-NET:", meta.loc[meta.dataset == "FG-NET", "sujeito"].nunique())
print("Imagens por sujeito no FG-NET (média):",
      round(meta[meta.dataset == "FG-NET"].groupby("sujeito").size().mean(), 1))

## 2. Qualidade dos dados

Checagens que justificam o tratamento:
- **valores ausentes** nos rótulos e nos embeddings
- **norma dos embeddings** (devem ser ~1, pois aplicamos normalização L2)
- **imagens duplicadas** (o UTKFace tem duplicatas conhecidas; a mesma foto em treino e teste inflaria as métricas)
- **rostos não detectados** no FG-NET (recorte central usado como alternativa)

In [ ]:
print("Ausentes em idade:", meta["idade"].isna().sum())
print("Ausentes nos embeddings:", int(np.isnan(E).sum()))
normas = np.linalg.norm(E, axis=1)
print(f"Norma dos embeddings: min={normas.min():.4f} max={normas.max():.4f}")
print("FG-NET sem rosto detectado:", int((~meta.loc[meta.dataset == 'FG-NET', 'rosto_detectado'].astype(bool)).sum()))

# Duplicatas: embeddings praticamente idênticos (arredondados) = mesma imagem
chave = pd.util.hash_pandas_object(pd.DataFrame(E.round(4)), index=False)
dup = chave.duplicated(keep=False)
meta["duplicada"] = dup.values
print(f"\nImagens em grupos duplicados: {dup.sum()}")
grupos = meta[dup].assign(chave=chave[dup].values).groupby("chave")["split"].agg(lambda s: tuple(sorted(set(s))))
vazamento = grupos[grupos.map(len) > 1]
print(f"Grupos de duplicatas espalhados em splits diferentes (vazamento): {len(vazamento)}")

**Decisão de tratamento:** remover duplicatas mantendo uma cópia por grupo, para evitar que a mesma foto apareça
em treino e teste. O resultado é salvo em `features_farl_limpo.parquet`, que passa a ser a base dos próximos notebooks.

In [ ]:
antes = len(df)
manter = ~chave.duplicated(keep="first").values
df_limpo = df[manter].reset_index(drop=True)
meta = meta[manter].reset_index(drop=True)
E = E[manter]
df_limpo.to_parquet(FEATURES_FILE.with_name("features_farl_limpo.parquet"), index=False)
print(f"Removidas {antes - len(df_limpo)} duplicatas. Base final: {len(df_limpo)} imagens")

## 3. Visualizações

### Fig. 1 · Distribuição de idade por dataset

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for nome, g in meta.groupby("dataset"):
    ax.hist(g["idade"], bins=range(0, 103), alpha=.65, density=True, label=f"{nome} (n={len(g)})", color=PALETA[nome])
ax.axvline(LIMIAR_PRINCIPAL, ls="--", c="k", lw=1); ax.text(LIMIAR_PRINCIPAL + 1, ax.get_ylim()[1] * .9, "18 anos")
ax.set(xlabel="Idade (anos)", ylabel="Densidade", title="Distribuição de idade por dataset")
ax.legend(); salvar("fig1_distribuicao_idade")

**Interpretação:** _(preencher após rodar)_ Comparar onde cada dataset concentra as idades. Esperado: UTKFace concentrado em adultos jovens
com um pico em bebês; FG-NET concentrado em crianças e adolescentes. Isso explica por que o FG-NET é um bom **teste externo para menores**.

### Fig. 2 · Proporção de menores em cada limiar legal

In [ ]:
linhas = []
for nome, g in meta.groupby("dataset"):
    for t in LIMIARES_CONTRIBUICAO:
        linhas.append({"dataset": nome, "limiar": f"< {t}", "pct": 100 * (g["idade"] < t).mean()})
prop = pd.DataFrame(linhas)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(prop, x="limiar", y="pct", hue="dataset", palette=PALETA, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.1f%%", fontsize=8)
ax.set(xlabel="Limiar de idade", ylabel="% de imagens abaixo do limiar",
       title="Desbalanceamento: proporção de menores por limiar")
salvar("fig2_proporcao_menores")

**Interpretação:** _(preencher)_ No UTKFace, quanto menor o limiar, mais rara a classe positiva. Isso justifica:
validação **estratificada**, métricas que não se enganam com desbalanceamento (**F2, recall**) e não usar acurácia como métrica principal.

### Fig. 3 · Zona de ambiguidade perto dos limiares (10 a 25 anos)

In [ ]:
zona = meta[(meta.idade >= 8) & (meta.idade <= 25)]
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.countplot(zona, x="idade", hue="dataset", palette=PALETA, ax=ax)
for t in [12, 13, 15, 16, 18, 21]:
    ax.axvline(t - 8, ls=":", c="gray", lw=1)
ax.set(xlabel="Idade (anos)", ylabel="Imagens", title="Quantidade de exemplos na faixa crítica (linhas = limiares legais)")
salvar("fig3_zona_critica")

**Interpretação:** _(preencher)_ A faixa de 13 a 17 anos é justamente a mais importante para o ECA Digital e,
em geral, uma das menos representadas. Poucos exemplos perto do limiar = fronteira de decisão mais incerta.

### Fig. 4 · Distribuição de idade por split (checagem da estratificação)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ordem = ["train", "val", "test", "test_externo"]
sns.boxplot(meta, x="split", y="idade", order=ordem, hue="split", palette="Blues", legend=False, ax=ax)
ax.set(xlabel="Split", ylabel="Idade (anos)", title="Idade por split: train, val e test devem ter a mesma forma")
salvar("fig4_idade_por_split")

**Interpretação:** _(preencher)_ Caixas iguais entre train/val/test confirmam que a estratificação funcionou: o modelo é avaliado
na mesma distribuição em que foi treinado. O `test_externo` (FG-NET) é propositalmente diferente.

### Fig. 5 · Composição demográfica do UTKFace (risco de viés)

In [ ]:
utk = meta[meta.dataset == "UTKFace"].copy()
utk["raca_nome"] = utk["raca"].map({0: "Branca", 1: "Negra", 2: "Asiática", 3: "Indiana", 4: "Outras"})
utk["genero_nome"] = utk["genero"].map({0: "Masculino", 1: "Feminino"})
utk["menor_18"] = np.where(utk["idade"] < 18, "Menor de 18", "Adulto")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
pd.crosstab(utk["raca_nome"], utk["menor_18"], normalize="index").mul(100).plot.barh(stacked=True, ax=axes[0], color=["#9BB7D4", "#2E6F9E"])
axes[0].set(xlabel="%", ylabel="", title="Menores vs adultos por grupo racial")
pd.crosstab(utk["genero_nome"], utk["menor_18"], normalize="index").mul(100).plot.barh(stacked=True, ax=axes[1], color=["#9BB7D4", "#2E6F9E"])
axes[1].set(xlabel="%", ylabel="", title="Menores vs adultos por gênero")
salvar("fig5_demografia")
print(utk["raca_nome"].value_counts(normalize=True).mul(100).round(1))

**Interpretação:** _(preencher)_ Se algum grupo tem bem menos crianças, o modelo aprende menos sobre menores desse grupo.
Isso entra na seção de **ética e limitações** e motiva avaliar o erro por grupo na fase de modelagem.

### Fig. 6 · Embeddings em 2D (PCA) coloridos por idade e por dataset

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=50, random_state=42).fit(E[meta.split == "train"])
Z = pca.transform(E)

rng = np.random.default_rng(0)
amostra = rng.choice(len(Z), size=min(6000, len(Z)), replace=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(Z[amostra, 0], Z[amostra, 1], c=meta.idade.values[amostra], cmap="viridis", s=4, alpha=.6)
plt.colorbar(sc, ax=axes[0], label="Idade")
axes[0].set(xlabel="PC1", ylabel="PC2", title="PCA dos embeddings FaRL, cor = idade")
for nome in ["UTKFace", "FG-NET"]:
    m = (meta.dataset.values == nome)
    idx = np.where(m)[0] if nome == "FG-NET" else rng.choice(np.where(m)[0], min(3000, m.sum()), replace=False)
    axes[1].scatter(Z[idx, 0], Z[idx, 1], s=4, alpha=.5, label=nome, c=PALETA[nome])
axes[1].set(xlabel="PC1", ylabel="PC2", title="PCA dos embeddings, cor = dataset"); axes[1].legend(markerscale=4)
salvar("fig6_pca_embeddings")

**Interpretação:** _(preencher)_ Um gradiente de cor no painel da esquerda mostra que a idade já está "codificada" nos embeddings,
mesmo sem treino (por isso o Ridge simples já teve MAE de ~5 anos). No painel da direita, se o FG-NET ocupa uma região diferente,
temos evidência visual da **mudança de domínio** que explica o MAE maior nele.

### Fig. 7 · Quantas dimensões os embeddings realmente usam

In [ ]:
acum = np.cumsum(pca.explained_variance_ratio_) * 100
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 51), acum, marker="o", ms=3)
for alvo in [50, 80]:
    k = int(np.argmax(acum >= alvo)) + 1 if (acum >= alvo).any() else None
    if k: ax.axhline(alvo, ls=":", c="gray"); ax.text(k + 1, alvo - 5, f"{alvo}% com {k} componentes")
ax.set(xlabel="Nº de componentes", ylabel="Variância explicada acumulada (%)", title="Variância explicada pelo PCA (50 primeiros componentes)")
salvar("fig7_variancia_pca")

**Interpretação:** _(preencher)_ Se poucas dimensões explicam muita variância, modelos sensíveis a dimensão (KNN, Árvore)
podem se beneficiar de PCA antes do treino. Isso vira uma decisão testável na fase de modelagem.

### Fig. 8 · Separabilidade menor vs adulto ao longo do PC mais relacionado com idade

In [ ]:
corr = [abs(np.corrcoef(Z[:, i], meta.idade)[0, 1]) for i in range(10)]
pc = int(np.argmax(corr))
plot = pd.DataFrame({"valor": Z[:, pc], "grupo": np.where(meta.idade < 18, "Menor de 18", "Adulto"), "dataset": meta.dataset})
fig, ax = plt.subplots(figsize=(9, 4))
sns.kdeplot(plot[plot.dataset == "UTKFace"], x="valor", hue="grupo", fill=True, common_norm=False, ax=ax)
ax.set(xlabel=f"PC{pc + 1} (|correlação com idade| = {corr[pc]:.2f})", ylabel="Densidade",
       title="Menores e adultos já se separam parcialmente em uma única direção do embedding")
salvar("fig8_separabilidade")

**Interpretação:** _(preencher)_ A sobreposição entre as curvas mostra a dificuldade do problema: uma única direção não basta,
por isso precisamos de modelos que combinem as 512 dimensões.

## 4. Resumo das decisões de tratamento

| Decisão | Motivo |
|---|---|
| Remover idades > 101 | O artigo usa 102 classes (0 a 101) |
| Ignorar arquivos com nome fora do padrão | Sem rótulo confiável |
| Recortar rosto no FG-NET (MTCNN) | O FaRL espera rosto recortado, como no UTKFace |
| Normalização L2 dos embeddings | Igual ao artigo ("normalized and projected output") |
| Remover duplicatas | Evitar vazamento entre treino e teste |
| Split estratificado por faixa etária, semente fixa | Mesma distribuição em train/val/test e reprodutibilidade |
| FG-NET só como teste externo | Medir generalização em outro domínio, rico em menores |
| Não usar acurácia como métrica principal | Classes desbalanceadas (Fig. 2) |